# Dr. Max: teste de consistência (5 takes)
Gera uma imagem de referência do personagem e 5 cenas usando IP-Adapter para manter o mesmo rosto e a mesma roupa.
Requer: **Accelerator = GPU T4** e **Internet = On**.

In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors

In [ ]:
import torch, os, shutil, time
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

MODEL = 'Lykon/dreamshaper-8'   # licença CreativeML OpenRAIL-M (permite uso comercial)
OUT = '/kaggle/working/cenas'
os.makedirs(OUT, exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, algorithm_type='dpmsolver++', final_sigmas_type='sigma_min', use_karras_sigmas=True)
pipe = pipe.to('cuda')
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ===== Estrutura de consistência =====
PERSONAGEM = ('friendly middle-aged male doctor, short neat brown hair, round black glasses, light stubble, '
              'white lab coat over light blue shirt, red tie, stethoscope around neck')
ESTILO = ('2d cartoon illustration, flat colors, bold outlines, cel shading, '
          'pastel background, high quality')
NEGATIVO = ('photo, realistic, 3d render, deformed, bad anatomy, extra fingers, mutated hands, extra limbs, '
            'blurry, low quality, jpeg artifacts, text, watermark, logo, signature, cropped head, duplicate')

# ===== Passo 1: imagem de referência (4 opções) =====
ref_prompt = f'{PERSONAGEM}, front view portrait, upper body, smiling, plain light background, {ESTILO}'
refs = []
for s in range(4):
    g = torch.Generator('cuda').manual_seed(100 + s)
    img = pipe(ref_prompt, negative_prompt=NEGATIVO, width=512, height=512,
               num_inference_steps=28, guidance_scale=7, generator=g).images[0]
    img.save(f'{OUT}/ref_{s}.png'); refs.append(img)

sheet = Image.new('RGB', (2048, 512), 'white')
for i, r in enumerate(refs): sheet.paste(r, (i * 512, 0))
sheet.save(f'{OUT}/refs_opcoes.png')
sheet.resize((1024, 256))

In [ ]:
# Escolha a melhor referência (0 a 3). Padrão: 0
REF_INDEX = 0
ref = refs[REF_INDEX]

pipe.load_ip_adapter('h94/IP-Adapter', subfolder='models', weight_name='ip-adapter-plus_sd15.safetensors')
pipe.set_ip_adapter_scale(0.6)

# ===== Passo 2: cenas (só muda câmera, ação e cenário) =====
cenas = [
    'medium shot, waving hello to the viewer, standing in a bright clinic office',
    'wide shot, sitting on a couch looking tired and sleepy, yawning, cozy living room in the afternoon',
    'close-up, explaining with one finger raised, friendly expression, clinic office background',
    'medium shot, pointing at a big alarm clock on the wall, bedroom background at night',
    'medium shot, smiling and giving a thumbs up, sunny park background',
]
W, H, SEED = 896, 512, 42
imgs = []
t0 = time.time()
for i, c in enumerate(cenas, 1):
    g = torch.Generator('cuda').manual_seed(SEED)
    p = f'{PERSONAGEM}, {c}, {ESTILO}'
    img = pipe(p, negative_prompt=NEGATIVO, ip_adapter_image=ref, width=W, height=H,
               num_inference_steps=28, guidance_scale=7, generator=g).images[0]
    img.save(f'{OUT}/{i:03d}.png'); imgs.append(img)
    print(f'{i:03d}.png ok')
print(f'{len(cenas)} cenas em {time.time()-t0:.0f}s')

In [ ]:
# ===== Folha de contato + zip =====
tw, th = 448, 256
contato = Image.new('RGB', (tw * 3, th * 2), 'white')
contato.paste(ref.resize((th, th)), (0, 0))
for k, im in enumerate(imgs, 1):
    contato.paste(im.resize((tw, th)), ((k % 3) * tw, (k // 3) * th))
contato.save('/kaggle/working/folha_contato.png')
shutil.make_archive('/kaggle/working/video_cenas', 'zip', OUT)
contato